# 🚀 NB 2 – YOLO11 Training in Google Colab
**Ultralytics YOLO11 | Google Drive | Resume | Fine-Tuning | Export**

| Abschnitt | Inhalt |
|-----------|--------|
| 1 | GPU-Prüfung |
| 2 | Installation |
| 3 | Google Drive einbinden |
| 4 | Datensatz laden (Link oder Drive) |
| 5 | Daten entpacken & prüfen |
| 6 | Konfiguration (Modell, Hyperparameter, Augmentation) |
| 7 | Training starten |
| 8 | Training fortsetzen (Resume) |
| 9 | Abschlussbericht & Charts |
| 10 | Validierung auf Testset |
| 11 | Inference – Objekte erkennen |
| 12 | Fine-Tuning mit neuem Datensatz |
| 13 | Modell exportieren |

> ⚠️ **Colab-Tipp:** Laufzeit → Laufzeittyp ändern → **T4 GPU** auswählen.

---

---
## ⚡ Abschnitt 1 – GPU-Prüfung

> Immer zuerst ausführen. Ohne GPU ist Training sehr langsam.
> Falls `CUDA verfügbar: False` → Laufzeit → Laufzeittyp ändern → T4 GPU.

In [ ]:
import torch, subprocess, sys

print('=' * 52)
print('  System-Check')
print('=' * 52)
print(f'  Python        : {sys.version.split()[0]}')
print(f'  PyTorch       : {torch.__version__}')
print(f'  CUDA verfügbar: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'  GPU           : {gpu.name}')
    print(f'  VRAM          : {gpu.total_memory / 1e9:.1f} GB')
    print('  ✅ GPU bereit – Training kann starten!')
else:
    print('  ⚠️  Kein GPU gefunden!')
    print('  → Laufzeit → Laufzeittyp ändern → T4 GPU')

print('=' * 52)
# NVIDA-Info
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                             '--format=csv,noheader'], capture_output=True, text=True)
    if result.returncode == 0:
        print(f'  nvidia-smi    : {result.stdout.strip()}')
except FileNotFoundError:
    pass


---
## 📦 Abschnitt 2 – Installation

Installiert Ultralytics (enthält YOLO11) und weitere Hilfspakete.
In Colab nur bei neuer Laufzeit nötig.

In [ ]:
# Ultralytics installieren (enthält YOLO11)
!pip install ultralytics --quiet

# Versionen prüfen
import ultralytics
from ultralytics import YOLO
ultralytics.checks()


---
## 📂 Abschnitt 3 – Google Drive einbinden

Bindet Google Drive ein. Damit werden:
- Trainingsergebnisse **automatisch in Drive gespeichert**
- Datensätze direkt aus Drive geladen
- Das fertige Modell in Drive abgelegt

> 💡 Nach dem Ausführen erscheint ein Popup zur Anmeldung.

In [ ]:
from google.colab import drive
import pathlib

drive.mount('/content/drive')

# Ordner in Drive anlegen (falls nicht vorhanden)
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/YOLO_Training')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'datasets').mkdir(exist_ok=True)
(DRIVE_ROOT / 'runs').mkdir(exist_ok=True)
(DRIVE_ROOT / 'models').mkdir(exist_ok=True)

print(f'✅ Google Drive eingebunden')
print(f'   Drive-Root : {DRIVE_ROOT}')
print(f'   Datasets   : {DRIVE_ROOT}/datasets')
print(f'   Runs       : {DRIVE_ROOT}/runs')
print(f'   Models     : {DRIVE_ROOT}/models')


---
## 📥 Abschnitt 4 – Datensatz laden

**Zwei Möglichkeiten – nur eine Zelle ausführen:**

- **4a** – Laden über direkten Download-Link (z.B. aus Nextcloud, Dropbox)
- **4b** – Laden aus Google Drive (empfohlen)

> Das ZIP muss die Struktur aus NB1 enthalten:
> `Data/images/train|val|test`, `Data/labels/train|val|test`, `Data/data.yaml`

In [ ]:
# ══════════════════════════════════════════════════
# ZELLE 4a – Laden über Download-Link
# Nur ausführen wenn kein Google Drive verwendet wird
# ══════════════════════════════════════════════════

DOWNLOAD_URL = 'https://DEIN-LINK-HIER/dataset.zip'  # ← anpassen

import urllib.request, pathlib

zip_path = pathlib.Path('/content/dataset.zip')
print(f'Lade Datensatz von: {DOWNLOAD_URL}')
urllib.request.urlretrieve(DOWNLOAD_URL, zip_path)
print(f'✅ Heruntergeladen: {zip_path.stat().st_size / 1e6:.1f} MB')


In [ ]:
# ══════════════════════════════════════════════════
# ZELLE 4b – Laden aus Google Drive (empfohlen)
# Vorher Abschnitt 3 ausführen!
# ══════════════════════════════════════════════════

import shutil, pathlib

# Pfad zur ZIP-Datei in Google Drive anpassen:
ZIP_IN_DRIVE = DRIVE_ROOT / 'datasets' / 'dataset.zip'  # ← anpassen

zip_path = pathlib.Path('/content/dataset.zip')

if not ZIP_IN_DRIVE.exists():
    print(f'⚠️  Datei nicht gefunden: {ZIP_IN_DRIVE}')
    print('   Bitte ZIP in Google Drive unter MyDrive/YOLO_Training/datasets/ ablegen.')
else:
    shutil.copy2(ZIP_IN_DRIVE, zip_path)
    print(f'✅ Kopiert: {zip_path.stat().st_size / 1e6:.1f} MB')
    print(f'   Quelle  : {ZIP_IN_DRIVE}')


---
## 📂 Abschnitt 5 – Daten entpacken & prüfen

Entpackt das ZIP und überprüft die YOLO-Ordnerstruktur.

In [ ]:
import zipfile, pathlib, shutil

zip_path    = pathlib.Path('/content/dataset.zip')
extract_tmp = pathlib.Path('/content/_zip_tmp')
DATA_DIR    = pathlib.Path('/content/Data')
IMG_EXT     = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Temporär entpacken und Struktur analysieren
if extract_tmp.exists(): shutil.rmtree(extract_tmp)
if DATA_DIR.exists():    shutil.rmtree(DATA_DIR)

print('Analysiere ZIP-Struktur ...')
with zipfile.ZipFile(zip_path, 'r') as z:
    names = z.namelist()
    # Zeige erste 8 Einträge
    for n in names[:8]: print(f'  {n}')
    if len(names) > 8: print(f'  ... ({len(names)} Dateien gesamt)')
    z.extractall(extract_tmp)

# data.yaml finden – egal wie tief im ZIP
yaml_candidates = list(extract_tmp.rglob('data.yaml'))
if not yaml_candidates:
    print('❌ data.yaml nicht gefunden im ZIP!')
    print('   Bitte NB1 erneut ausführen und ZIP neu erstellen.')
    raise FileNotFoundError('data.yaml fehlt im ZIP')

# Wurzel des Datensatzes = Ordner der data.yaml
dataset_root = yaml_candidates[0].parent
print(f'\n✅ Datensatz-Root gefunden: {dataset_root}')

# Nach /content/Data verschieben
shutil.copytree(dataset_root, DATA_DIR)
shutil.rmtree(extract_tmp)
print(f'✅ Entpackt nach: {DATA_DIR}')

# data.yaml Pfad aktualisieren
yaml_path = DATA_DIR / 'data.yaml'

# Pfad in data.yaml korrigieren (zeigt noch auf lokalen Windows-Pfad)
yaml_text = yaml_path.read_text(encoding='utf-8')
yaml_text_fixed = ''
for line in yaml_text.splitlines():
    if line.strip().startswith('path:'):
        yaml_text_fixed += f'path: {DATA_DIR}\n'
    else:
        yaml_text_fixed += line + '\n'
yaml_path.write_text(yaml_text_fixed, encoding='utf-8')
print(f'\n✅ data.yaml Pfad auf Colab angepasst:')
print(yaml_text_fixed)

# Strukturcheck
print('=' * 50)
print('  Strukturcheck')
print('=' * 50)
total_i = total_l = 0
for split in ['train', 'val', 'test']:
    img_dir = DATA_DIR / 'images' / split
    lbl_dir = DATA_DIR / 'labels' / split
    if not img_dir.exists():
        print(f'  ⚠️   {split}: nicht vorhanden (übersprungen)')
        continue
    imgs = [p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXT]
    lbls = [p for p in lbl_dir.iterdir() if p.suffix=='.txt'] if lbl_dir.exists() else []
    total_i += len(imgs)
    total_l += len(lbls)
    print(f'  ✅  {split:6s}  {len(imgs):4d} Bilder  |  {len(lbls):4d} Labels')
print('=' * 50)
print(f'  Gesamt: {total_i} Bilder  |  {total_l} Labels')

DATA_YAML = str(yaml_path)
print(f'\n✅ DATA_YAML = {DATA_YAML}')


---
## ⚙️ Abschnitt 6 – Konfiguration

> 🔧 **Nur hier anpassen** – alle Trainingsparameter an einer Stelle.

### Modellgrößen (YOLO11)

| Modell | Parameter | Geschwindigkeit | Genauigkeit | Empfehlung |
|--------|-----------|-----------------|-------------|------------|
| `yolo11n.pt` | 2.6M | ⚡⚡⚡⚡ | ⭐⭐ | Einstieg, schnelles Testen |
| `yolo11s.pt` | 9.4M | ⚡⚡⚡ | ⭐⭐⭐ | Guter Kompromiss |
| `yolo11m.pt` | 20M | ⚡⚡ | ⭐⭐⭐⭐ | Empfohlen für Produktion |
| `yolo11l.pt` | 25M | ⚡ | ⭐⭐⭐⭐⭐ | Hohe Genauigkeit |
| `yolo11x.pt` | 56M | 🐢 | ⭐⭐⭐⭐⭐ | Maximum Genauigkeit |

In [ ]:
import pathlib

# ╔══════════════════════════════════════════════════════════╗
# ║  KONFIGURATION – hier anpassen                           ║
# ╚══════════════════════════════════════════════════════════╝

# ── Modell ──────────────────────────────────────────────────
MODEL_SIZE   = 'yolo11n.pt'   # yolo11n / s / m / l / x
PROJECT_NAME = 'yolo_bauteile' # Name des Projekts
RUN_NAME     = 'run_1'         # Name des Trainingslaufs

# ── Training Grundparameter ─────────────────────────────────
EPOCHS       = 100             # Anzahl Trainingsepochen
PATIENCE     = 20              # Early Stop: Epochen ohne Verbesserung
BATCH        = 16              # Batch-Größe (-1 = auto)
IMG_SIZE     = 640             # Bildgröße in Pixel (640 = Standard)
WORKERS      = 4               # Datenlade-Threads
SEED         = 42              # Zufallsseed für Reproduzierbarkeit

# ── Optimierung ─────────────────────────────────────────────
OPTIMIZER    = 'auto'          # auto / SGD / Adam / AdamW
LR0          = 0.01            # Initiale Lernrate
LRF          = 0.01            # Finale Lernrate (Faktor von LR0)
MOMENTUM     = 0.937           # SGD Momentum / Adam beta1
WEIGHT_DECAY = 0.0005          # L2-Regularisierung
WARMUP_EPOCHS = 3.0            # Warmup-Epochen

# ── Augmentation ────────────────────────────────────────────
# Geometrisch
FLIPLR       = 0.5             # Horizontales Spiegeln (0.0-1.0)
FLIPUD       = 0.0             # Vertikales Spiegeln
DEGREES      = 5.0             # Rotation ±Grad
TRANSLATE    = 0.1             # Verschiebung (Anteil Bildgröße)
SCALE        = 0.5             # Skalierung ±Faktor
SHEAR        = 2.0             # Scherung ±Grad
PERSPECTIVE  = 0.0             # Perspektivverzerrung (0-0.001)
# Farbe
HSV_H        = 0.015           # Farbton-Variation
HSV_S        = 0.7             # Sättigung-Variation
HSV_V        = 0.4             # Helligkeit-Variation
# Spezial
MOSAIC       = 1.0             # Mosaic-Augmentation (0.0-1.0)
MIXUP        = 0.0             # MixUp-Augmentation (0.0-1.0)
COPY_PASTE   = 0.0             # Copy-Paste-Augmentation
ERASING      = 0.4             # Random Erasing

# ── Speicherorte ────────────────────────────────────────────
COLAB_RUNS   = pathlib.Path('/content/runs')
DRIVE_RUNS   = DRIVE_ROOT / 'runs'

print('=' * 52)
print(f'  Modell      : {MODEL_SIZE}')
print(f'  Projekt     : {PROJECT_NAME}/{RUN_NAME}')
print(f'  Epochen     : {EPOCHS}  |  Early Stop: {PATIENCE}')
print(f'  Batch       : {BATCH}  |  ImgSize: {IMG_SIZE}')
print(f'  Lernrate    : {LR0} → {LR0*LRF:.5f}')
print(f'  data.yaml   : {DATA_YAML}')
print('=' * 52)


---
## 🏋️ Abschnitt 7 – Training starten

Startet das Training mit den Parametern aus Abschnitt 6.
Ergebnisse werden in Colab **und** in Google Drive gespeichert.

> ⏱️ Richtwert: ~1-3 Minuten pro Epoche auf T4 GPU (abhängig von Datenmenge).

> ⚠️ Falls das Training unterbrochen wird → **Abschnitt 8 (Resume)** verwenden.

In [ ]:
from ultralytics import YOLO
import shutil, pathlib, time

# Modell laden (pretrained)
model = YOLO(MODEL_SIZE)
print(f'✅ Modell geladen: {MODEL_SIZE}')

start = time.time()

# Training starten
results = model.train(
    data         = DATA_YAML,
    project      = str(COLAB_RUNS / PROJECT_NAME),
    name         = RUN_NAME,
    # Grundparameter
    epochs       = EPOCHS,
    patience     = PATIENCE,
    batch        = BATCH,
    imgsz        = IMG_SIZE,
    workers      = WORKERS,
    seed         = SEED,
    # Optimierung
    optimizer    = OPTIMIZER,
    lr0          = LR0,
    lrf          = LRF,
    momentum     = MOMENTUM,
    weight_decay = WEIGHT_DECAY,
    warmup_epochs= WARMUP_EPOCHS,
    # Augmentation (geometrisch)
    fliplr       = FLIPLR,
    flipud       = FLIPUD,
    degrees      = DEGREES,
    translate    = TRANSLATE,
    scale        = SCALE,
    shear        = SHEAR,
    perspective  = PERSPECTIVE,
    # Augmentation (Farbe)
    hsv_h        = HSV_H,
    hsv_s        = HSV_S,
    hsv_v        = HSV_V,
    # Augmentation (Spezial)
    mosaic       = MOSAIC,
    mixup        = MIXUP,
    copy_paste   = COPY_PASTE,
    erasing      = ERASING,
    # Sonstiges
    device       = 0,      # 0 = erste GPU
    exist_ok     = True,   # Ordner überschreiben
    pretrained   = True,   # Pretrained Weights verwenden
    verbose      = True,
    save         = True,
    save_period  = 10,     # Checkpoint alle N Epochen
)

dauer = time.time() - start
print(f'\n✅ Training abgeschlossen in {dauer/60:.1f} Minuten')

# Ergebnisse in Google Drive sichern
run_dir  = pathlib.Path(results.save_dir)
drive_ziel = DRIVE_RUNS / PROJECT_NAME / RUN_NAME
if drive_ziel.exists():
    shutil.rmtree(drive_ziel)
shutil.copytree(run_dir, drive_ziel)
print(f'✅ Ergebnisse in Drive gesichert:')
print(f'   {drive_ziel}')

# Bestes Modell speichern
best_pt     = run_dir / 'weights' / 'best.pt'
drive_model = DRIVE_ROOT / 'models' / f'{PROJECT_NAME}_{RUN_NAME}_best.pt'
shutil.copy2(best_pt, drive_model)
print(f'✅ Bestes Modell: {drive_model}')

BEST_MODEL = str(best_pt)
RUN_DIR    = str(run_dir)


---
## 🔄 Abschnitt 8 – Training fortsetzen (Resume)

Falls das Training unterbrochen wurde (Colab-Timeout, manuell gestoppt)
kann es hier nahtlos fortgesetzt werden.

**Zwei Szenarien:**
- **8a** – Checkpoint liegt noch in Colab (Laufzeit noch aktiv)
- **8b** – Checkpoint aus Google Drive wiederherstellen (neue Laufzeit)

In [ ]:
# ══════════════════════════════════════════════════
# ZELLE 8a – Resume aus Colab (Laufzeit noch aktiv)
# ══════════════════════════════════════════════════
from ultralytics import YOLO
import pathlib, shutil, time

# Letzten Checkpoint finden
last_pt = pathlib.Path(COLAB_RUNS) / PROJECT_NAME / RUN_NAME / 'weights' / 'last.pt'

if not last_pt.exists():
    print(f'⚠️  Checkpoint nicht gefunden: {last_pt}')
    print('   → Zelle 8b verwenden (aus Drive laden)')
else:
    print(f'✅ Checkpoint gefunden: {last_pt}')
    model   = YOLO(str(last_pt))
    start   = time.time()
    results = model.train(resume=True)
    dauer   = time.time() - start
    print(f'\n✅ Training fortgesetzt und abgeschlossen in {dauer/60:.1f} Minuten')

    # In Drive sichern
    run_dir    = pathlib.Path(results.save_dir)
    drive_ziel = DRIVE_RUNS / PROJECT_NAME / RUN_NAME
    if drive_ziel.exists(): shutil.rmtree(drive_ziel)
    shutil.copytree(run_dir, drive_ziel)
    BEST_MODEL = str(run_dir / 'weights' / 'best.pt')
    RUN_DIR    = str(run_dir)
    print(f'✅ Gesichert: {drive_ziel}')


In [ ]:
# ══════════════════════════════════════════════════
# ZELLE 8b – Resume aus Google Drive (neue Laufzeit)
# Vorher Abschnitt 3 (Drive einbinden) ausführen!
# ══════════════════════════════════════════════════
from ultralytics import YOLO
import pathlib, shutil, time

# Checkpoint aus Drive nach Colab kopieren
drive_last = DRIVE_RUNS / PROJECT_NAME / RUN_NAME / 'weights' / 'last.pt'
colab_last = pathlib.Path('/content/last.pt')

if not drive_last.exists():
    print(f'⚠️  Kein Checkpoint in Drive: {drive_last}')
else:
    shutil.copy2(drive_last, colab_last)
    print(f'✅ Checkpoint aus Drive geladen: {colab_last}')

    model   = YOLO(str(colab_last))
    start   = time.time()
    results = model.train(resume=True)
    dauer   = time.time() - start
    print(f'\n✅ Training abgeschlossen in {dauer/60:.1f} Minuten')

    run_dir    = pathlib.Path(results.save_dir)
    drive_ziel = DRIVE_RUNS / PROJECT_NAME / RUN_NAME
    if drive_ziel.exists(): shutil.rmtree(drive_ziel)
    shutil.copytree(run_dir, drive_ziel)
    BEST_MODEL = str(run_dir / 'weights' / 'best.pt')
    RUN_DIR    = str(run_dir)
    print(f'✅ Gesichert: {drive_ziel}')


---
## 📊 Abschnitt 9 – Abschlussbericht & Charts

Zeigt die wichtigsten Trainingsmetriken und Kurven:
- Loss-Kurven (Box, Cls, DFL)
- Precision / Recall / mAP über Epochen
- Konfusionsmatrix
- Ergebnis-Übersicht

In [ ]:
import pathlib, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

run_dir = pathlib.Path(RUN_DIR)

# ── Metriken aus results.csv ─────────────────────────────────
csv_path = run_dir / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    best_idx = df['metrics/mAP50(B)'].idxmax()
    best     = df.iloc[best_idx]

    print('=' * 55)
    print('  TRAININGSERGEBNIS – Beste Epoche')
    print('=' * 55)
    print(f'  Epoche         : {int(best["epoch"])+1} / {len(df)}')
    print(f'  mAP50          : {best["metrics/mAP50(B)"]:.4f}')
    print(f'  mAP50-95       : {best["metrics/mAP50-95(B)"]:.4f}')
    print(f'  Precision      : {best["metrics/precision(B)"]:.4f}')
    print(f'  Recall         : {best["metrics/recall(B)"]:.4f}')
    print(f'  Train Box Loss : {best["train/box_loss"]:.4f}')
    print(f'  Val   Box Loss : {best["val/box_loss"]:.4f}')
    print('=' * 55)

    # ── Charts ────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f'Training: {PROJECT_NAME}/{RUN_NAME}',
                 fontsize=14, fontweight='bold')

    plots = [
        ('train/box_loss',          'val/box_loss',         'Box Loss'),
        ('train/cls_loss',          'val/cls_loss',         'Cls Loss'),
        ('train/dfl_loss',          'val/dfl_loss',         'DFL Loss'),
        ('metrics/precision(B)',    None,                   'Precision'),
        ('metrics/recall(B)',       None,                   'Recall'),
        ('metrics/mAP50(B)',        'metrics/mAP50-95(B)',  'mAP'),
    ]

    for ax, (col1, col2, titel) in zip(axes.flatten(), plots):
        if col1 in df.columns:
            ax.plot(df['epoch'], df[col1], label='train' if col2 else col1.split('/')[-1])
        if col2 and col2 in df.columns:
            ax.plot(df['epoch'], df[col2], label='val' if 'loss' in col1 else 'mAP50-95',
                    linestyle='--')
        ax.axvline(best_idx, color='red', linestyle=':', alpha=0.5, label='best')
        ax.set_title(titel, fontweight='bold')
        ax.set_xlabel('Epoche')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    display(fig)
    plt.close()

# ── Konfusionsmatrix ─────────────────────────────────────────
for cm_name in ['confusion_matrix_normalized.png', 'confusion_matrix.png']:
    cm_path = run_dir / cm_name
    if cm_path.exists():
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(mpimg.imread(str(cm_path)))
        ax.axis('off')
        ax.set_title('Konfusionsmatrix', fontweight='bold')
        display(fig)
        plt.close()
        break


---
## 🧪 Abschnitt 10 – Validierung auf Testset

Wertet das beste Modell auf dem **Testset** aus.
Das Testset war während des Trainings komplett unberührt.

| Metrik | Bedeutung |
|--------|----------|
| **mAP50** | Mean Average Precision bei IoU=0.50 |
| **mAP50-95** | mAP über IoU 0.50–0.95 (strenger) |
| **Precision** | Anteil korrekte Vorhersagen |
| **Recall** | Anteil gefundene Objekte |

In [ ]:
from ultralytics import YOLO

model = YOLO(BEST_MODEL)
print(f'Validiere: {BEST_MODEL}\n')

metrics = model.val(
    data   = DATA_YAML,
    split  = 'test',     # auf Testset validieren
    imgsz  = IMG_SIZE,
    device = 0,
    verbose= True,
)

print()
print('=' * 45)
print('  VALIDIERUNG – Testset')
print('=' * 45)
print(f'  mAP50     : {metrics.box.map50:.4f}')
print(f'  mAP50-95  : {metrics.box.map:.4f}')
print(f'  Precision : {metrics.box.mp:.4f}')
print(f'  Recall    : {metrics.box.mr:.4f}')
print('=' * 45)

# Pro Klasse
if hasattr(metrics.box, 'ap_class_index'):
    names = model.names
    print('\n  Pro Klasse:')
    for idx, ap50 in zip(metrics.box.ap_class_index,
                          metrics.box.ap50):
        print(f'    {names[idx]:<15} mAP50={ap50:.4f}')


---
## 🔍 Abschnitt 11 – Inference

Erkennt Objekte auf eigenen Bildern mit dem trainierten Modell.

**Zwei Möglichkeiten:**
- **11a** – Bilder direkt hochladen
- **11b** – Zufällige Bilder aus dem Validierungs- oder Testset

In [ ]:
# ══════════════════════════════════════════════════
# ZELLE 11a – Eigene Bilder hochladen
# ══════════════════════════════════════════════════
from google.colab import files
from ultralytics import YOLO
import pathlib, matplotlib.pyplot as plt, matplotlib.image as mpimg
from IPython.display import display

# Bilder hochladen
uploaded = files.upload()
upload_dir = pathlib.Path('/content/inference_input')
upload_dir.mkdir(exist_ok=True)
for name, data in uploaded.items():
    (upload_dir / name).write_bytes(data)

model   = YOLO(BEST_MODEL)
results = model.predict(
    source    = str(upload_dir),
    imgsz     = IMG_SIZE,
    conf      = 0.25,    # Konfidenz-Schwellenwert
    iou       = 0.7,     # IoU-Schwellenwert (NMS)
    save      = True,
    save_conf = True,
    project   = '/content/inference_out',
    name      = 'upload',
    exist_ok  = True,
)

# Ergebnisse anzeigen
out_dir = pathlib.Path('/content/inference_out/upload')
bilder  = sorted(out_dir.glob('*.jpg'))[:6]
n_cols  = min(3, len(bilder))
n_rows  = -(-len(bilder) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7*n_cols, 5*n_rows))
import numpy as np
axes = np.array(axes).flatten()
for ax, p in zip(axes, bilder):
    ax.imshow(mpimg.imread(str(p)))
    ax.set_title(p.name, fontsize=8)
    ax.axis('off')
for ax in axes[len(bilder):]: ax.axis('off')
plt.tight_layout()
display(fig)
plt.close()


In [ ]:
# ══════════════════════════════════════════════════
# ZELLE 11b – Zufällige Bilder aus Val/Test
# ══════════════════════════════════════════════════
from ultralytics import YOLO
import pathlib, random, matplotlib.pyplot as plt, matplotlib.image as mpimg
import matplotlib.patches as patches
import numpy as np
from IPython.display import display

SPLIT_CHECK = 'val'   # 'val' oder 'test'
ANZAHL_INF  = 6
CONF_THRESH = 0.25

img_dir = pathlib.Path('/content/Data/images') / SPLIT_CHECK
IMG_EXT = {'.jpg','.jpeg','.png','.bmp','.webp'}
alle    = [p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXT]
auswahl = random.sample(alle, min(ANZAHL_INF, len(alle)))

model   = YOLO(BEST_MODEL)
n_cols  = min(3, len(auswahl))
n_rows  = -(-len(auswahl) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7*n_cols, 5*n_rows))
axes    = np.array(axes).flatten()

for ax, img_path in zip(axes, auswahl):
    result = model.predict(str(img_path), conf=CONF_THRESH,
                           imgsz=IMG_SIZE, verbose=False)[0]
    img = mpimg.imread(str(img_path))
    H, W = img.shape[:2]
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'{img_path.name}  [{len(result.boxes)} Obj.]', fontsize=8)
    FARBEN = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']
    for box in result.boxes:
        x1,y1,x2,y2 = box.xyxy[0].tolist()
        cid   = int(box.cls[0])
        conf  = float(box.conf[0])
        farbe = FARBEN[cid % len(FARBEN)]
        name  = model.names[cid]
        ax.add_patch(patches.Rectangle(
            (x1,y1), x2-x1, y2-y1,
            linewidth=2, edgecolor=farbe, facecolor=farbe+'33'))
        ax.text(x1+3, y1-6, f'{name} {conf:.2f}',
                color='white', fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc=farbe, alpha=0.85, ec='none'))

for ax in axes[len(auswahl):]: ax.axis('off')
plt.tight_layout()
display(fig)
plt.close()


---
## 🔬 Abschnitt 12 – Fine-Tuning mit neuem Datensatz

Wenn das Modell eine Klasse noch nicht gut erkennt (z.B. `Taster`),
kann es mit einem **erweiterten Datensatz** nachtrainiert werden.

**Workflow:**
1. Neue Bilder aufnehmen und in Label Studio annotieren
2. In NB0 + NB1 verarbeiten → neues ZIP erstellen
3. Neues ZIP in Drive laden
4. Diese Zellen ausführen → Modell wird vom besten Checkpoint weiter trainiert

> 💡 **Lerntransfer:** Das Modell startet vom `best.pt` des vorherigen Trainings.
> Es muss nicht von Null lernen – nur die neuen Muster werden verfeinert.

> ⚙️ **Empfohlene Einstellungen beim Fine-Tuning:**
> - Weniger Epochen (z.B. 30–50)
> - Kleinere Lernrate (z.B. `LR0 = 0.001`)
> - Gleiche Bildgröße wie beim ersten Training

In [ ]:
from ultralytics import YOLO
import shutil, pathlib, time

# ── Einstellungen Fine-Tuning ─────────────────────────────────
FT_ZIP_DRIVE  = DRIVE_ROOT / 'datasets' / 'dataset_v2.zip'  # ← neues ZIP
FT_RUN_NAME   = 'run_2_finetune'
FT_EPOCHS     = 50
FT_LR0        = 0.001          # kleinere Lernrate!
FT_PATIENCE   = 15

# ── Neues ZIP entpacken ───────────────────────────────────────
ft_zip = pathlib.Path('/content/dataset_v2.zip')
shutil.copy2(FT_ZIP_DRIVE, ft_zip)
ft_data = pathlib.Path('/content/Data_v2')
if ft_data.exists(): shutil.rmtree(ft_data)
import zipfile
with zipfile.ZipFile(ft_zip) as z:
    z.extractall('/content/Data_v2')
FT_YAML = str(ft_data / 'data.yaml')
print(f'✅ Neuer Datensatz entpackt: {ft_data}')

# ── Fine-Tuning starten (von best.pt) ────────────────────────
model = YOLO(BEST_MODEL)   # startet vom besten vorherigen Checkpoint
print(f'   Ausgangsmodell: {BEST_MODEL}')

start   = time.time()
results = model.train(
    data         = FT_YAML,
    project      = str(COLAB_RUNS / PROJECT_NAME),
    name         = FT_RUN_NAME,
    epochs       = FT_EPOCHS,
    patience     = FT_PATIENCE,
    batch        = BATCH,
    imgsz        = IMG_SIZE,
    lr0          = FT_LR0,
    lrf          = 0.01,
    pretrained   = True,
    exist_ok     = True,
    device       = 0,
)
dauer = time.time() - start
print(f'\n✅ Fine-Tuning abgeschlossen in {dauer/60:.1f} Minuten')

# In Drive sichern
ft_run_dir  = pathlib.Path(results.save_dir)
drive_ziel  = DRIVE_RUNS / PROJECT_NAME / FT_RUN_NAME
if drive_ziel.exists(): shutil.rmtree(drive_ziel)
shutil.copytree(ft_run_dir, drive_ziel)

ft_best     = ft_run_dir / 'weights' / 'best.pt'
drive_model = DRIVE_ROOT / 'models' / f'{PROJECT_NAME}_{FT_RUN_NAME}_best.pt'
shutil.copy2(ft_best, drive_model)

BEST_MODEL = str(ft_best)   # für Abschnitt 13 verwenden
RUN_DIR    = str(ft_run_dir)
print(f'✅ Fine-Tuned Modell: {drive_model}')


---
## 💾 Abschnitt 13 – Modell exportieren

### Übersicht Exportformate

| Format | Verwendung | Geschwindigkeit | Kompatibilität |
|--------|-----------|-----------------|----------------|
| **`.pt`** | PyTorch – Standard, Streamlit, Python | ⚡⚡⚡ | Python |
| **ONNX** | Plattformunabhängig, viele Frameworks | ⚡⚡⚡ | Sehr breit |
| **TFLite** | Android, Raspberry Pi, Microcontroller | ⚡⚡⚡⚡ | Mobile |
| **TensorRT** | NVIDIA-Optimierung für maximale GPU-Geschwindigkeit | ⚡⚡⚡⚡⚡ | NVIDIA only |
| **CoreML** | Apple-Geräte (iPhone, Mac) | ⚡⚡⚡⚡ | Apple only |
| **OpenVINO** | Intel CPU/GPU Optimierung | ⚡⚡⚡⚡ | Intel |

**Für Streamlit → `.pt` verwenden** (Standard, keine Konvertierung nötig).

> 💡 ONNX ist die beste Wahl wenn das Modell auf verschiedenen Plattformen laufen soll.

In [ ]:
from ultralytics import YOLO
import shutil, pathlib
from google.colab import files

# ── Einstellungen ─────────────────────────────────────────────
# Format wählen: 'torchscript' | 'onnx' | 'tflite' | 'engine' (TensorRT)
EXPORT_FORMAT = 'pt'    # Standard: .pt für Streamlit

model = YOLO(BEST_MODEL)

if EXPORT_FORMAT == 'pt':
    # .pt braucht keinen Export – direkt kopieren
    export_path = pathlib.Path(BEST_MODEL)
    print(f'✅ .pt Modell bereit: {export_path}')
else:
    print(f'Exportiere als {EXPORT_FORMAT} ...')
    export_path = model.export(
        format  = EXPORT_FORMAT,
        imgsz   = IMG_SIZE,
        device  = 0,
        half    = False,   # True = FP16 (schneller, etwas ungenauer)
    )
    export_path = pathlib.Path(export_path)
    print(f'✅ Export: {export_path}')

# In Drive speichern
drive_export = DRIVE_ROOT / 'models' / export_path.name
shutil.copy2(export_path, drive_export)
print(f'✅ In Drive gespeichert: {drive_export}')
print(f'   Größe: {export_path.stat().st_size / 1e6:.1f} MB')

# Herunterladen
print('\nStarte Download ...')
files.download(str(export_path))


---
## 🏁 NB 2 abgeschlossen – Checkliste

- ☐ GPU verfügbar (T4) *(Abschnitt 1)*
- ☐ Datensatz geladen und entpackt *(Abschnitt 4+5)*
- ☐ Training abgeschlossen *(Abschnitt 7)*
- ☐ Ergebnisse in Google Drive gesichert *(Abschnitt 7)*
- ☐ Abschlussbericht & Charts geprüft *(Abschnitt 9)*
- ☐ Validierung auf Testset *(Abschnitt 10)*
- ☐ Inference getestet *(Abschnitt 11)*
- ☐ Modell als `.pt` heruntergeladen *(Abschnitt 13)*

---
🚀 **Weiter mit NB 3:** Streamlit-App zum Testen des Modells